# Duckify Simulation notebook

### Connect to the simulator / to the robot

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from URBasic import Joint6D, TCP6D

In [ ]:
SIMULATION = True

In [ ]:
from duckify_simulation.duckify_sim import DuckifySim

# # Connect to the simulator
duckify_sim = DuckifySim()
print("Connected to simulator")

In [ ]:
from URBasic import ISCoin

In [ ]:
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
if not SIMULATION:
    iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

In [ ]:
robot_sim = duckify_sim.robot_control
robot_true = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```txt
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [ ]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    tcp_home = robot_true.get_actual_tcp_pose()

In [ ]:
robot_sim.movej(home)
if SIMULATION:
    tcp_home = robot_sim.get_actual_tcp_pose()

In [ ]:
print(f"Home TCP: {tcp_home}")

In [ ]:
# target_tcp = TCP6D.createFromMetersRadians(
#       0.0,      # x
#      -0.35,     # y
#       0.20,     # z
#       0.0,      # rx
#       3.14,      # ry
#       0.0       # rz
#   )

In [ ]:
# current_joints = robot.get_actual_joint_positions()
# target_joints = robot.get_inverse_kin(target_tcp, qnear=current_joints)

## Set TCP

In [ ]:
if not SIMULATION:
    from src.calibration import collect_data
    NUM_MEASURES = 20

    tcps = collect_data(robot_true, NUM_MEASURES)
else:
    tcps = [
        [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
        [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
        [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
        [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
        [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
        [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
    ]

In [ ]:
from src.calibration import get_tcp_offset, validate_calibration
tcp_offset = get_tcp_offset(tcps)
robot_sim.set_tcp(tcp_offset)
if not SIMULATION:
    robot_true.set_tcp(tcp_offset)    
motion_val_cal = validate_calibration(robot_sim)

In [ ]:
for m in motion_val_cal:
    robot_sim.movel(m)

In [ ]:
if not SIMULATION:
    robot_true.freedrive_mode()
    input("Put the robot in position for validate calibration.")
    robot_true.end_freedrive_mode()

    motion_val_cal = validate_calibration(robot_true)

    input("Ready to move?")
    for m in motion_val_cal:
        robot_true.movel(m)

## Compute Transformation object2robot

In [ ]:
file_path = "trapezoid_letters-trapezoid_letters-trace.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data["generated_at"])

---- transformation function ----

In [ ]:
def _normal_to_rxyz(n):
    x,y,z = n
    z = -z
    rx = np.arctan2(-y, np.sqrt(x**2 + z**2))
    ry = np.arctan2(x, z)
    rz = 0
    return [rx,ry,rz]

In [ ]:
def create_transformation(A, B):
    A = np.asarray(A)[:, :3]
    B = np.asarray(B)[:, :3]
    assert A.shape == B.shape
    N = A.shape[0]

    # Build matrix for affine solve: [A | 1]
    P = np.hstack([A, np.ones((N, 1))])  # Nx4

    # Solve P @ X = B, where X is 4×3 (R^T and t)
    X, _, _, _ = np.linalg.lstsq(P, B, rcond=None)

    # Extract R and t
    R = X[:3, :].T   # 3×3
    t = X[3, :]      # 3

    # Build full 4×4 affine matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t

    # Normal transform
    R_normal = np.linalg.inv(R).T
    T_normal = np.eye(4)
    T_normal[:3, :3] = R_normal

    def AtoB(p):
        p = np.asarray(p)
        point = p[:3]
        normal = p[3:]

        # Transform point
        p_new = T @ [*point, 1]

        # Transform normal
        nx,ny,nz = normal
        n_new = T_normal @ [nx,ny,nz, 1]
        n_new /= np.linalg.norm(n_new)
        r_new = _normal_to_rxyz(n_new[:3])

        return [*p_new[:3], *r_new]

    return AtoB


---- Create transformation function ----

In [ ]:
from src.transformation import collect_data as collect_data_tf

p_world = np.array(data["calibration"])

if len(p_world) < 4:
    print("Missing point")

if not SIMULATION:
    p_tcp = collect_data_tf(robot_true, p_world)
else:
    p_tcp = np.array([
        [-0.03389773, -0.2829937,   0.09458801],# -0.24982057, -3.09150359, -0.18723581],
        [-0.03366482, -0.31970029,  0.09458314],# -0.28493231, -3.12244751, -0.07563827],
        [ 0.02527303, -0.32058601,  0.09427673],#  0.21952408,  3.00241046,  0.09098521],
        [ 0.05483491, -0.26206449,  0.05878293]#,  0.18852376, -2.79599659, -0.21783004]
    ])

In [ ]:
obj2robot = create_transformation(p_world, p_tcp)

In [ ]:
# Example
faces = [
    [0.0, 0.9999999999999999, 0.0],
    [0.0, 0.4999999425479707, 0.8660254369543807],
    [0.0, 0.500000078148597, -0.8660253586653204],
    [0.7660444804970836, 0.6427875651410451, 0.0],
    [-0.7660444410855507, 0.6427876121098819, 4.820907090824115e-08]
]
p_w = [0,0,0]
for f in faces:
    t = obj2robot([*p_w, *f])
    print(t)

## Generate trajectory

---- transformation function ----

In [ ]:
def _rotvec_to_rotmat(r):
    theta = np.linalg.norm(r)
    if theta < 1e-12:
        return np.eye(3)
    k = r / theta
    K = np.array([[0, -k[2], k[1]],
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(theta)*K + (1-np.cos(theta))*(K @ K)

def _rotmat_to_rotvec(R):
    theta = np.arccos((np.trace(R) - 1) / 2)
    if theta < 1e-12:
        return np.zeros(3)
    rx = (R[2,1] - R[1,2]) / (2*np.sin(theta))
    ry = (R[0,2] - R[2,0]) / (2*np.sin(theta))
    rz = (R[1,0] - R[0,1]) / (2*np.sin(theta))
    return theta * np.array([rx, ry, rz])

def tcp_trans(tcp1, tcp2):
    # Décomposition
    p1 = np.array(tcp1[:3])
    r1 = np.array(tcp1[3:])
    p2 = np.array(tcp2[:3])
    r2 = np.array(tcp2[3:])

    # Matrices de rotation
    R1 = _rotvec_to_rotmat(r1)
    R2 = _rotvec_to_rotmat(r2)

    # Composition
    p_new = p1 + R1 @ p2
    R_new = R1 @ R2
    r_new = _rotmat_to_rotvec(R_new)

    return np.concatenate([p_new, r_new])

---- Create trajectory ----

In [ ]:
# from src.transformation import tcp_trans
traces = data["traces"]
draw_motions = []
move_motions = []
above = [0,0,-0.02,0,0,0]
for trace in traces:
    normal = trace["face"]
    normal[2] = - normal[2]
    t_world = trace["path"]
    motion = []
    
    # Drawing motion
    for i, p in enumerate(t_world):
        p_w = (*p, *normal)
        x,y,z,rx,ry,rz = obj2robot(p_w)
        
        # Add entry point
        if i == 0:
            x_t,y_t,z_t,_,_,_ = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            motion.append(m)

        # Add drawing point
        m = TCP6D.createFromMetersRadians(x,y,z, rx,ry,rz)
        motion.append(m)

        # Add exit point
        if i == len(t_world)-1:
            x_t,y_t,z_t,_,_,_ = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            motion.append(m)
    
    if motion:
        draw_motions.append(motion)
        move_motions.append((motion[0][0], motion[-1][-1]))
    #break

In [ ]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([home, move_motions[m][0]])
    else:
        move_outside.append([move_motions[m-1][-1], home, move_motions[m][0]])
move_outside.append([move_motions[-1][-1], home])

## Drawing algorithm

### Drawing algorithm simulation

In [ ]:
robot_sim.movej(home)
for k in range(len(move_motions)):
    
    # Move in position
    print(data["traces"][k]["face"])
    for m in move_outside[k]:
        if isinstance(m, TCP6D):
            robot_sim.movel(m)
            print(m)
        
        elif isinstance(m, Joint6D):
            robot_sim.movej(m)
            print(robot_sim.get_actual_tcp_pose())
    
    print()

    # Draw on object
    #for m in draw_motions[k]:
    #    robot_sim.movel_waypoints(m)

print("Return home")
for m in move_outside[-1]:
    if isinstance(m, TCP6D):
        robot_sim.movel(m)
        print(m)
    elif isinstance(m, Joint6D):
        robot_sim.movej(m)
        print(robot_sim.get_actual_tcp_pose())


### Drawing algorithm test move to position

The robot will move around the object to reach positions.

```text
                   home -> input position
output position -> home -> input position
output position -> home
```

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    for k in range(len(move_motions)):
        
        # Move in position
        print(data["traces"][k]["face"])
        for m in move_outside[k]:
            if isinstance(m, TCP6D):
                robot_true.movel(m)
                print(m)
            
            elif isinstance(m, Joint6D):
                robot_true.movej(m)
                print(robot_true.get_actual_tcp_pose())
        
        print()

        # Draw on object
        #for m in draw_motions[k]:
        #    robot_true.movel_waypoints(m)

    print("Return home")
    for m in move_outside[-1]:
        if isinstance(m, TCP6D):
            robot_true.movel(m)
            print(m)
        elif isinstance(m, Joint6D):
            robot_true.movej(m)
            print(robot_true.get_actual_tcp_pose())

### Drawing algorithm test drawing

The robot will move around the object and draw on it.

```text
                   home -> input position -> draw
output position -> home -> input position -> draw
output position -> home
```

In [ ]:
if not SIMULATION:
    robot_true.movej(home)
    for k in range(len(move_motions)):
        
        # Move in position
        print(data["traces"][k]["face"])
        for m in move_outside[k]:
            if isinstance(m, TCP6D):
                robot_true.movel(m)
                print(m)
            
            elif isinstance(m, Joint6D):
                robot_true.movej(m)
                print(robot_true.get_actual_tcp_pose())
        
        print()

        # Draw on object
        for m in draw_motions[k]:
            robot_true.movel_waypoints(m)

    print("Return home")
    for m in move_outside[-1]:
        if isinstance(m, TCP6D):
            robot_true.movel(m)
            print(m)
        elif isinstance(m, Joint6D):
            robot_true.movej(m)
            print(robot_true.get_actual_tcp_pose())